# 3 Relaxation: AlN

Rather than fixing the structure and only relaxing the electronic degrees
of freedom (as in every other section of this tutorial), Abinit can also
relax the atomic positions and/or the unit cell itself, driven by the
computed forces and stresses. This is controlled mainly through `ionmov`
(the ionic relaxation algorithm) and `optcell` (whether/how the cell shape
and volume are allowed to change), with convergence controlled by
`tolmxf` (max force) or `tolrff`, instead of the `tolvrs`/`tolwfr` used for
a fixed-geometry SCF run.

Below, we relax AlN (`mp-661`) starting from its experimental (Materials
Project) structure -- already run for you (`flow_aln_relax/`, next to this
notebook). Standalone version: `../Examples/make_aln_relax.py`.


In [ ]:
wlib.print_source(wlib.aln_relax_input)
wlib.print_source(wlib.build_aln_relax_flow)

### Build and run the flow

Standalone version: `../Examples/make_aln_relax.py`. From a terminal, in
`Examples/` (so the resulting `flow_*/` directory lands next to the
script, matching the paths used below):

```
cd ../Examples
python make_aln_relax.py
abirun.py flow_aln_relax scheduler
```

This has already been run for you (`flow_aln_relax/`, next to this
notebook) -- the analysis below reads directly from it.


`ionmov=2` is a Broyden algorithm for the atomic positions; `optcell=1`
additionally lets the cell *volume* relax isotropically (`acell` is
dilated, while `rprim` -- the cell shape -- is left unchanged). Convergence
is judged on the max force (`tolmxf`) rather than the potential residual
(`tolvrs`) used for fixed-geometry runs, and `strfact` rescales the stress
so it's comparable in magnitude to the forces.

`out_HIST.nc`, produced alongside the usual `GSR`, records the structure,
energy, forces and stress at every relaxation step:


In [ ]:
with abilab.abiopen("flow_aln_relax/w0/t0/outdata/out_HIST.nc") as hist:
    hist.plot()
    print("Initial lattice (a, b, c):", hist.initial_structure.lattice.abc)
    print("Relaxed lattice (a, b, c):", hist.final_structure.lattice.abc)

`HistFile` records the whole relaxation trajectory (energy, forces,
stress and the structure itself, at every ionic step) -- `.plot()` above
shows all of it converging, and `.initial_structure`/`.final_structure`
give direct access to the endpoints. There's no `workshop_lib` function
for what comes next -- `../Examples/save_aln_structure.py` is a plain
post-processing script, not a flow builder -- but here's what it does:
grab the *final* relaxed structure from the task's `GSR` file:


In [ ]:
flow = flowtk.Flow.from_file("flow_aln_relax")
task = flow[0][0]                         # the only task of the only Work
gsr_path = task.outdir.has_abiext("GSR")   # retrieve the output GSR file

with abilab.abiopen(gsr_path) as gsr:
    relaxed_structure = gsr.structure

print(relaxed_structure)

`../Examples/save_aln_structure.py` does exactly this, then writes the
result to a standalone `.cif`:

```
cd ../Examples
python save_aln_structure.py
```

writes `Examples/Data/AlN_relaxed.cif` -- the relaxed structure, ready to
be loaded back with `abilab.Structure.from_file(...)` and reused as the
input structure of a later calculation (e.g. a band structure or phonon
flow at the relaxed volume, instead of the experimental one).


In [ ]:
wlib.print_source(wlib._bandstructure_inputs)
wlib.print_source(wlib.build_gaas_ebands_flow)

### Build and run the flow

Standalone version: `../Examples/make_gaas_ebands.py`. From a terminal,
in `Examples/` (so the resulting `flow_*/` directory lands next to the
script, matching the paths used below):

```
cd ../Examples
python make_gaas_ebands.py
abirun.py flow_gaas_ebands scheduler
```

This has already been run for you (`flow_gaas_ebands/`, next to this
notebook) -- the analysis below reads directly from it.


In [ ]:
with abilab.abiopen("flow_gaas_ebands/w0/t1/outdata/out_GSR.nc") as gsr:
    gaas_ebands = gsr.ebands

fig = gaas_ebands.plot(color="b", show=False)
fig.gca().set_ylim(-10, 10)
fig.gca().set_title("GaAs");

GaAs has a direct gap at $\Gamma$, while silicon's fundamental gap
(Notebook 1) is indirect. This is a good moment to discuss why LDA/GGA
systematically underestimate these gaps, and what the standard corrections
are ($GW$, hybrid functionals, scissor operators).

> **Note.** `ElectronBands` objects also expose `.plot_with_edos(edos)` to
> overlay the density of states next to the bands, and `to_bxsf()` /
> `.plot_fermi_surface()` for metals -- not needed for GaAs/Si, but worth
> knowing about.

Continue with [`3-Command_line_interface.ipynb`](3-Command_line_interface.ipynb).